# Transformer Reinforcement Learning (TRL)

To oversimply the life-cycle and execution of RLHF and Fine-tuning over language models, we will use TRL hugging face library to see how this can performed easily with a high level of abstraction thanks to this library. As all hugging face at the beggining, it can be overwhelming and extremely high level; to avoid this and ending up using its API without understanding anything, i find really helpful to dissect high level libraries (like langchain, llamaindex, ragas...) into its main modules and the algorithms that are encapsulated:

1. HuggingFace Transformers: This is the base hugginface transformers library that has several utilities
2. HuggingFace TRL: 

In [ ]:
"""
hhrlhf_1000_rlhf_ppo_lora_cpu_with_plots.py

Same pipeline as before (1k subset RLHF PPO + LoRA) but with plotting:
- logs per-step metrics (reward, kl, losses)
- plots Reward, KL, and Loss curves and saves them as PNGs

Designed to run on CPU (~16GB RAM) with the same small config.
"""
import os
import random
import math
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
)

# TRL + PEFT imports
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model
from peft import LoraConfig, get_peft_model

# plotting
import matplotlib.pyplot as plt
from collections import deque

# ---------------------------
# CONFIG: small for CPU
# ---------------------------
SEED = 1234
random.seed(SEED)
torch.manual_seed(SEED)

NUM_EXAMPLES = 1000
RM_EPOCHS = 3
PPO_EPOCHS = 2
QUERIES_FOR_PPO = 200
GEN_MAX_NEW_TOKENS = 64
BATCH_SIZE_RM = 16
MODEL_NAME = "gpt2"
DEVICE = torch.device("cpu")

# ---------------------------
# (A) Load dataset subset
# ---------------------------
def load_preference_subset(n_examples=NUM_EXAMPLES):
    preferred_ids = ["Dahoas/full-hh-rlhf", "Anthropic/hh-rlhf"]
    ds = None; used_id = None
    for did in preferred_ids:
        try:
            print(f"Trying to load dataset {did} ...")
            ds_full = load_dataset(did, split=f"train[:{n_examples}]")
            ds = ds_full; used_id = did
            break
        except Exception as e:
            print(f"Could not load {did}: {e}")
    if ds is None:
        raise RuntimeError("Failed to load any known HH-RLHF dataset id.")
    print(f"Loaded dataset subset {used_id} with {len(ds)} examples.")
    return ds, used_id

# ---------------------------
# (B) Build RM triples
# ---------------------------
def build_rm_pairs(ds):
    pairs = []
    for ex in ds:
        if "prompt" in ex and "chosen" in ex and "rejected" in ex:
            prompt = ex["prompt"]; chosen = ex["chosen"]; rejected = ex["rejected"]
        elif "prompt" in ex and "chosen_completion" in ex and "rejected_completion" in ex:
            prompt = ex["prompt"]; chosen = ex["chosen_completion"]; rejected = ex["rejected_completion"]
        elif "instruction" in ex and "response0" in ex and "response1" in ex:
            prompt = ex.get("instruction",""); chosen = ex.get("response0",""); rejected = ex.get("response1","")
            if "label" in ex:
                lab = ex["label"]
                if lab == 1:
                    chosen, rejected = ex.get("response1",""), ex.get("response0","")
        else:
            keys = [k for k,v in ex.items() if isinstance(v, str) and len(v)>10]
            if len(keys) >= 3:
                prompt, chosen, rejected = ex[keys[0]], ex[keys[1]], ex[keys[2]]
            else:
                continue
        if not (isinstance(prompt,str) and isinstance(chosen,str) and isinstance(rejected,str)):
            continue
        if len(prompt) < 5 or len(chosen) < 3 or len(rejected) < 3:
            continue
        pairs.append((prompt.strip(), chosen.strip(), rejected.strip()))
    print(f"Constructed {len(pairs)} RM triples.")
    return pairs

# ---------------------------
# (C) Reward model dataset & train
# ---------------------------
class RewardDataset(Dataset):
    def __init__(self, triples):
        self.examples = []
        for prompt, chosen, rejected in triples:
            self.examples.append((prompt + " ||| " + chosen, 1.0))
            self.examples.append((prompt + " ||| " + rejected, 0.0))
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        txt, lbl = self.examples[idx]
        return {"text": txt, "label": torch.tensor(lbl, dtype=torch.float32)}

def train_reward_model(triples, device=DEVICE, epochs=RM_EPOCHS):
    print("Training reward model...")
    ds = RewardDataset(triples)
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=1)
    model.to(device)

    def collate(batch):
        texts = [b["text"] for b in batch]
        labels = torch.stack([b["label"] for b in batch]).unsqueeze(-1).to(device)
        enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256)
        enc = {k: v.to(device) for k, v in enc.items()}
        enc["labels"] = labels
        return enc

    loader = DataLoader(ds, batch_size=BATCH_SIZE_RM, shuffle=True, collate_fn=collate)
    opt = optim.AdamW(model.parameters(), lr=3e-5)
    loss_fn = nn.MSELoss()

    model.train()
    for ep in range(epochs):
        total = 0.0; n = 0
        for batch in loader:
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            preds = torch.sigmoid(logits)
            loss = loss_fn(preds, batch["labels"])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n += 1
        print(f"[RM] epoch {ep+1}/{epochs} loss={total/max(1,n):.4f}")
    model.eval()
    return tokenizer, model

@torch.no_grad()
def score_with_rm(rm_tokenizer, rm_model, prompt, response, device=DEVICE):
    txt = prompt + " ||| " + response
    enc = rm_tokenizer(txt, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    logits = rm_model(**enc).logits.squeeze(-1)
    return float(torch.sigmoid(logits).item())

# ---------------------------
# (D) Policy + LoRA setup
# ---------------------------
def setup_policy_and_ref(tokenizer_name=MODEL_NAME, model_name=MODEL_NAME, device=DEVICE):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(tokenizer_name)
    if tok.pad_token is None:
        tok.add_special_tokens({"pad_token": "<|pad|>"})

    policy = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)
    policy.resize_token_embeddings(len(tok))

    ref_model = create_reference_model(policy)

    peft_config = LoraConfig(
        task_type="CAUSAL_LM",
        inference_mode=False,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
    )
    policy = get_peft_model(policy, peft_config)
    policy.to(device); ref_model.to(device)
    return tok, policy, ref_model

# ---------------------------
# (E) PPO loop with logging & plotting data collection
# ---------------------------
def run_small_ppo_and_log(tok, policy, ref_model, rm_tok, rm_model, queries, device=DEVICE):
    print("Setting up PPOTrainer (tiny config)...")
    ppo_config = PPOConfig(
        model_name=MODEL_NAME,
        learning_rate=1.4e-5,
        batch_size=1,
        forward_batch_size=1,
        ppo_epochs=1,
        seed=SEED,
        kl_coef=0.05,
    )
    trainer = PPOTrainer(config=ppo_config, model=policy, ref_model=ref_model, tokenizer=tok, dataset=None)

    gen_kwargs = {
        "max_new_tokens": GEN_MAX_NEW_TOKENS,
        "do_sample": True,
        "top_k": 50,
        "top_p": 0.95,
        "temperature": 1.0,
        "pad_token_id": tok.pad_token_id,
        "eos_token_id": tok.eos_token_id,
    }

    queries = queries[:min(len(queries), QUERIES_FOR_PPO)]
    print(f"Running PPO on {len(queries)} prompts.")

    # collectors
    rewards = []
    kl_vals = []
    policy_losses = []
    value_losses = []
    pg_losses = []
    step_idx = 0

    # moving average helper
    def moving_avg(xs, k=25):
        if len(xs) == 0: return []
        window = deque(maxlen=k)
        res = []
        s = 0.0
        for x in xs:
            window.append(x)
            res.append(sum(window)/len(window))
        return res

    for epoch in range(PPO_EPOCHS):
        print(f"=== PPO epoch {epoch+1}/{PPO_EPOCHS} ===")
        for q in queries:
            enc = tok(q, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
            with torch.no_grad():
                out = trainer.model.generate(**enc, **gen_kwargs)
            gen_text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            reward = score_with_rm(rm_tok, rm_model, q, gen_text, device)

            # do the PPO step and get stats
            stats = trainer.step([q], [gen_text], [reward])

            # try to extract values from stats dict robustly
            kl = stats.get("kl_loss") or stats.get("ref_mean_kl") or stats.get("mean_kl") or None
            pl = stats.get("policy_loss") or stats.get("pg_loss") or stats.get("loss") or None
            vl = stats.get("value_loss") or stats.get("vf_loss") or None
            # store
            rewards.append(reward)
            kl_vals.append(float(kl) if kl is not None else float("nan"))
            policy_losses.append(float(pl) if pl is not None else float("nan"))
            value_losses.append(float(vl) if vl is not None else float("nan"))
            # record pg_losses if available
            if "pg_loss" in stats:
                pg_losses.append(float(stats["pg_loss"]))
            step_idx += 1

            # small logging
            print(f"step={step_idx} reward={reward:.3f} kl={kl} policy_loss={pl} value_loss={vl}")

    # After training, create plots
    print("Creating plots...")

    os.makedirs("plots", exist_ok=True)

    # 1) Reward per step + rolling average
    plt.figure()
    plt.plot(rewards)
    ma = moving_avg(rewards, k=25)
    if ma:
        plt.plot(ma)
    plt.title("Reward per step (and rolling average)")
    plt.xlabel("Step")
    plt.ylabel("Reward")
    plt.grid(True)
    rpath = os.path.join("plots", "reward_per_step.png")
    plt.savefig(rpath)
    print("Saved", rpath)
    plt.show()

    # 2) KL per step + rolling average
    plt.figure()
    plt.plot(kl_vals)
    ma_kl = moving_avg([v for v in kl_vals if not math.isnan(v)], k=25)
    if ma_kl:
        # we need to align length; simple approach: plot moving avg same length as kl_vals but with nans padded
        full_ma_kl = []
        idx = 0
        for v in kl_vals:
            if math.isnan(v):
                full_ma_kl.append(float("nan"))
            else:
                full_ma_kl.append(ma_kl[idx]); idx += 1
        plt.plot(full_ma_kl)
    plt.title("KL (per step) and rolling average")
    plt.xlabel("Step")
    plt.ylabel("KL")
    plt.grid(True)
    kpath = os.path.join("plots", "kl_per_step.png")
    plt.savefig(kpath)
    print("Saved", kpath)
    plt.show()

    # 3) Policy/value losses per step (if present)
    plt.figure()
    # convert nan lists to numeric arrays for plotting
    def safe_plot(xs, label):
        if all(math.isnan(x) for x in xs):
            return False
        plt.plot([x if not math.isnan(x) else float("nan") for x in xs], label=label)
        return True

    plotted_any = False
    if safe_plot(policy_losses, "policy_loss"):
        plotted_any = True
    if safe_plot(value_losses, "value_loss"):
        plotted_any = True
    if safe_plot(pg_losses, "pg_loss"):
        plotted_any = True

    if plotted_any:
        plt.title("Losses per step (when available)")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        lpath = os.path.join("plots", "losses_per_step.png")
        plt.savefig(lpath)
        print("Saved", lpath)
        plt.show()
    else:
        print("No loss values were available in trainer stats to plot.")

    # return the trainer and the collected metrics (in case user wants them)
    return trainer, {
        "rewards": rewards,
        "kl": kl_vals,
        "policy_loss": policy_losses,
        "value_loss": value_losses,
        "pg_loss": pg_losses,
    }

# ---------------------------
# (F) Main
# ---------------------------
def main():
    device = DEVICE
    print("Device:", device)
    ds, used_id = load_preference_subset(NUM_EXAMPLES)
    triples = build_rm_pairs(ds)
    if len(triples) < 50:
        raise RuntimeError("Not enough triples; dataset schema may not match expectations.")

    rm_tok, rm_model = train_reward_model(triples, device=device, epochs=RM_EPOCHS)
    tok, policy, ref_model = setup_policy_and_ref(model_name=MODEL_NAME, device=device)
    queries = [t[0] for t in triples]
    queries = list(dict.fromkeys(queries))

    trainer, metrics = run_small_ppo_and_log(tok, policy, ref_model, rm_tok, rm_model, queries, device=device)

    print("Done. Plots saved in ./plots/ directory.")
    # Save metrics as a small csv for later use
    import csv
    out_csv = "plots/training_metrics.csv"
    keys = ["step", "reward", "kl", "policy_loss", "value_loss", "pg_loss"]
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(keys)
        n = len(metrics["rewards"])
        for i in range(n):
            row = [
                i+1,
                metrics["rewards"][i],
                metrics["kl"][i] if i < len(metrics["kl"]) else "",
                metrics["policy_loss"][i] if i < len(metrics["policy_loss"]) else "",
                metrics["value_loss"][i] if i < len(metrics["value_loss"]) else "",
                metrics["pg_loss"][i] if i < len(metrics["pg_loss"]) else "",
            ]
            writer.writerow(row)
    print("Saved CSV to", out_csv)

    # Save LoRA adapters
    save_dir = "./lora_adapters_hh_1000_with_plots"
    trainer.model.save_pretrained(save_dir)
    print("Saved LoRA adapters to", save_dir)

if __name__ == "__main__":
    main()
